In [24]:
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.utils import save_image, make_grid
import numpy as np
import matplotlib.pyplot as plt
import os
import math
from model import Ganerator_MNIST, VAE_Encoder_MNIST, LabelCondLatentDenoiser
from com_utils import array_to_bits, bits_to_array

In [16]:
batch_size = 64
z_dim = 64
device = "cuda"
t_val=300         # 給 DM 評估用
lam=0.5          # Fashion latent 加到 MNIST latent 的係數
do_whiten=True     # 是否白化 z_f
out_dir="./save_data/test_sys"

ds = datasets.MNIST(root=r"D:\Python\SemCom\saved_data", train=True, download=True, transform=lambda img: np.array(img, dtype=np.uint8))
ds_f = datasets.FashionMNIST(root=r"D:\Python\SemCom\saved_data", train=True, download=True, transform=lambda img: np.array(img, dtype=np.uint8))
dl = DataLoader(ds, batch_size=batch_size, shuffle=True, num_workers=0, drop_last=True)
dl_f = DataLoader(ds_f, batch_size=batch_size, shuffle=True, num_workers=0, drop_last=True)


In [33]:
for i, ((x_m, _), (x_f, _)) in enumerate(zip(dl, dl_f)):

    # x_m: [B, 28, 28], uint8, 0~255
    x_m_uint8 = x_m[:25]
    x_f_uint8 = x_f[:25]

    # if i == 0:
    #     print(x_m_uint8.dtype)
    #     print(x_m_uint8.shape)
    #     print(x_m_uint8.min(), x_m_uint8.max())

    # 轉 bits
    bits_m = array_to_bits(x_m_uint8, bit_width=8)
    bits_f = array_to_bits(x_f_uint8, bit_width=8)
    # print(x_m_uint8)
    # print(x_m_uint8.dtype)
    print(bits_m.shape)
    # print(x_m_uint8.min(), x_m_uint8.max())
    # print(torch.unique(x_m_uint8))
    break
    rec_m = bits_to_array(bits_m, 8)
    rec_f = bits_to_array(bits_f, 8)
    rec_m_img = rec_m.reshape(25, 28, 28)   # (25,28,28)
    rec_m_img = torch.tensor(rec_m_img).unsqueeze(1).float() / 255.0
    save_image(rec_m_img, "./save_data/test_traditional/recon_MM.png", nrow=5)
    recon_grid_m = make_grid(rec_m_img, nrow=5, normalize=True)
    plt.figure(figsize=(5,5))
    plt.imshow(recon_grid_m.permute(1, 2, 0).cpu())
    plt.title(f"Batch {i}")
    plt.axis("off")
    plt.show(block=False)
    plt.pause(1.5)   # 顯示1.5秒
    plt.close()
    break
    # print(bits_m.shape)

    # 如果要存圖，要轉成 float [0,1]，並補 channel 維度
    x_m_img = x_m_uint8.float().unsqueeze(1) / 255.0
    x_f_img = x_f_uint8.float().unsqueeze(1) / 255.0

    save_image(x_m_img, f"./save_data/test_traditional/input_MNIST_{i}.png", nrow=5)
    save_image(x_f_img, f"./save_data/test_traditional/input_fashion_{i}.png", nrow=5)
    
    grid_m = make_grid(x_m_img, nrow=5)
    grid_f = make_grid(x_f_img, nrow=5)

    plt.figure(figsize=(5,5))
    plt.imshow(grid_m.permute(1, 2, 0).numpy(), cmap="gray")
    plt.title(f"Batch {i}")
    plt.axis("off")
    plt.show(block=False)
    plt.pause(1.5)   # 顯示1.5秒
    plt.close()
    plt.figure(figsize=(5,5))
    plt.imshow(grid_f.permute(1, 2, 0).numpy(), cmap="gray")
    plt.title(f"Batch {i}")
    plt.axis("off")
    plt.show(block=False)
    plt.pause(1.5)   # 顯示1.5秒
    plt.close()
    break

(156800,)


In [ ]:
G = Ganerator_MNIST(z_dim=z_dim).to(device)
G.load_state_dict(torch.load("./save_model/G_wgangp_C.pth"))
G.eval()
for p in G.parameters():
    p.requires_grad = False

E = VAE_Encoder_MNIST(latent_dim=z_dim).to(device)
E.load_state_dict(torch.load("./save_model/E_invert_to_G_C_only.pth"))
E.eval()
for p in E.parameters():
    p.requires_grad = False

E_f = VAE_Encoder_MNIST(latent_dim=z_dim).to(device)
E_f.load_state_dict(torch.load("./save_model/E_invert_to_G_F.pth"))
E_f.eval()
for p in E_f.parameters():
    p.requires_grad = False

DM = LabelCondLatentDenoiser(num_classes=2, z_dim=z_dim, tdim=128, ydim=64).to(device)
DM.load_state_dict(torch.load("./save_model/DM_latent_label.pth"))
DM.eval()
for p in DM.parameters():
    p.requires_grad = False